# Graph Algorithms: Centrality, Community, and More

## Overview

Run graph algorithms to discover insights in your data. This notebook covers native Ryugraph algorithms and NetworkX integration for advanced analytics.

**What you'll learn:**
- Run PageRank, Connected Components, and Louvain community detection
- Use NetworkX algorithms (betweenness, degree, closeness centrality)
- Query algorithm results and combine multiple measures
- Manage algorithm locks

**Prerequisites:**
- Completed: [04_core_exploring_schemas.ipynb](./04_core_exploring_schemas.ipynb)
- Running instance with graph data

**Time to complete:** 15 minutes

---

**Test Coverage:**
- Native Ryugraph algorithms (PageRank, WCC, SCC, Louvain, K-Core)
- NetworkX algorithms (betweenness, degree, closeness centrality)
- Algorithm introspection (list, info)
- Lock status management

**Note:** Algorithm tests write computed properties to nodes using `algo_` prefix.

In [ ]:
import os

# Parameters cell - papermill will inject values here
EXPECTED_NODE_COUNT = 5

# OPTIMIZATION: If SNAPSHOT_ID is provided, skip snapshot creation (~60s saved)
# The shared snapshot is created once at session start and reused by all tests
SNAPSHOT_ID = None  # Will be injected by papermill when running via pytest

## Setup

### Test Data Isolation Strategy

1. **Self-Contained**: Creates its own mapping, snapshot, and instance.
2. **Unique Property Names**: All computed properties use `algo_*` prefix to avoid conflicts with other tests.
3. **Cleanup**: Automatic cleanup via cleanup framework.

In [ ]:
import os
import sys

print(f"Python version: {sys.version}")
print(f"GRAPH_OLAP_API_URL: {os.environ.get('GRAPH_OLAP_API_URL', 'not set')}")
print(f"Expected node count: {EXPECTED_NODE_COUNT}")

In [ ]:
from graph_olap import notebook
from graph_olap.models import EdgeDefinition, NodeDefinition
from graph_olap.models.mapping import PropertyDefinition
from graph_olap.testing import TestPersona
from graph_olap_schemas import WrapperType

# Wake up Starburst Galaxy cluster (auto-suspends after 5 min idle)
notebook.wake_starburst()

# Create test context with automatic cleanup
ctx = notebook.test("AlgoTest", persona=TestPersona.ANALYST_ALICE)
client = ctx.client

# Check if using shared snapshot (optimization)
if SNAPSHOT_ID:
    print(f"OPTIMIZATION: Using shared snapshot {SNAPSHOT_ID} (skipping ~60s of snapshot creation)")
    # Get mapping ID from the shared snapshot
    shared_snapshot = client.snapshots.get(int(SNAPSHOT_ID))
    MAPPING_ID = shared_snapshot.mapping_id
    print(f"  Mapping ID from shared snapshot: {MAPPING_ID}")
else:
    # Define test data - same Person/KNOWS schema as smoke test
    person_node = NodeDefinition(
        label="Person",
        sql="SELECT DISTINCT id, name, age FROM bigquery.graph_olap_e2e.persons",
        primary_key={"name": "id", "type": "STRING"},
        properties=[
            PropertyDefinition(name="name", type="STRING"),
            PropertyDefinition(name="age", type="INT64"),
        ]
    )

    knows_edge = EdgeDefinition(
        type="KNOWS",
        from_node="Person",
        to_node="Person",
        sql="SELECT DISTINCT from_id, to_id, since FROM bigquery.graph_olap_e2e.friendships",
        from_key="from_id",
        to_key="to_id",
        properties=[PropertyDefinition(name="since", type="INT64")],
    )

print(f"Test run ID: {ctx.run_id}")
print(f"Creating test resources with prefix: AlgoTest-*-{ctx.run_id}")

## Start Cleanup Context Manager

All resources created from here will be automatically cleaned up.

In [ ]:
# Resources are automatically tracked and cleaned up via ctx
print("Starting Algorithm E2E Test - resources will be cleaned up automatically via atexit")

In [ ]:
# Create mapping using SDK directly (skip if using shared snapshot)
if SNAPSHOT_ID:
    print(f"Skipping mapping creation - using shared snapshot {SNAPSHOT_ID}")
    MAPPING_NAME = f"SharedE2EMapping"  # Name from conftest.py shared_snapshot
else:
    mapping = ctx.mapping(
        description="Mapping for algorithm E2E tests",
        node_definitions=[person_node],
        edge_definitions=[knows_edge],
    )
    MAPPING_ID = mapping.id
    MAPPING_NAME = mapping.name

    print(f"Created mapping: {MAPPING_NAME} (id={MAPPING_ID})")

In [ ]:
# SNAPSHOT FUNCTIONALITY DISABLED
# Create instance directly from mapping or use shared snapshot
if SNAPSHOT_ID:
    print(f"Skipping snapshot creation - using shared snapshot {SNAPSHOT_ID}")
    SNAPSHOT_NAME = "SharedE2ESnapshot"  # Name from conftest.py shared_snapshot
    # SNAPSHOT_ID is already set from parameters
else:
    # Create instance directly from mapping using create_from_mapping()
    # The snapshot is created automatically in the background
    print(f"Creating instance directly from mapping (snapshot created automatically)...")
    print("  (This may take ~3-4 minutes for snapshot export + instance startup)")
    
    instance = client.instances.create_from_mapping_and_wait(
        mapping_id=MAPPING_ID,
        name=f"AlgoTest-Instance-{ctx.run_id}",
        wrapper_type=WrapperType.RYUGRAPH,
        timeout=300,  # Longer timeout for snapshot export
        poll_interval=5,
    )
    INSTANCE_ID = instance.id
    SNAPSHOT_ID = instance.snapshot_id
    SNAPSHOT_NAME = f"auto-snapshot-{SNAPSHOT_ID}"
    ctx._track('instance', INSTANCE_ID, instance.name)
    
    print(f"Created instance: {instance.name} (id={INSTANCE_ID}, status={instance.status})")
    print(f"Auto-created snapshot: (id={SNAPSHOT_ID})")

In [ ]:
# Create instance (only if using shared snapshot - otherwise already created above)
if 'instance' not in dir() or instance is None:
    # Using shared snapshot - need to create instance from it
    snapshot_obj = client.snapshots.get(SNAPSHOT_ID)
    
    print(f"Creating instance from shared snapshot... (spawns wrapper pod)")
    
    instance = client.instances.create_and_wait(
        snapshot_id=SNAPSHOT_ID,
        name=f"AlgoTest-Instance-{ctx.run_id}",
        wrapper_type=WrapperType.RYUGRAPH,
        timeout=180,
        poll_interval=5,
    )
    INSTANCE_ID = instance.id
    ctx._track('instance', INSTANCE_ID, instance.name)

INSTANCE_NAME = instance.name

print(f"Instance ready: {INSTANCE_NAME} (id={INSTANCE_ID}, status={instance.status})")
print(f"Instance URL: {instance.instance_url}")

In [ ]:
# Connect to instance via SDK
conn = ctx.connect(instance)
print(f"Connected to instance {INSTANCE_ID}")

## Track Algorithm Properties for Cleanup

Track all properties that will be written by algorithms so they can be cleaned up automatically.

In [ ]:
# Track ALL algorithm properties that will be written during tests
# This will be cleaned up at the end
algo_properties = [
    # Native algorithm properties
    "algo_generic_wcc",     # Test 2.1: Generic run() for WCC
    "algo_pr",              # Test 2.2: PageRank
    "algo_wcc",             # Test 2.4: Connected Components
    "algo_community",       # Test 2.5: Louvain
    "algo_scc",             # Test 2.6: SCC
    "algo_scc_ko",          # Test 2.7: SCC Kosaraju
    "algo_kcore",           # Test 2.8: K-Core
    "algo_label_prop",      # Test 2.9: Label Propagation
    "algo_triangles",       # Test 2.10: Triangle Count
    # NetworkX algorithm properties
    "algo_bc",              # Test 3.3: Betweenness Centrality
    "algo_dc",              # Test 3.4: Degree Centrality
    "algo_cc",              # Test 3.5: Closeness Centrality
    "algo_nx_generic_dc",   # Test 3.6: Generic NetworkX run()
    "algo_ev",              # Test 3.9: Eigenvector Centrality
    "algo_clustering",      # Test 3.10: Clustering Coefficient
    # Lock status test properties
    "algo_lock_test_pr",    # Test 4.2: Lock release test
]

# Track for cleanup - will be handled specially in cleanup loop
ctx._track('graph_properties', conn, {'node_label': 'Person', 'property_names': algo_properties})

print("Algorithm properties tracked for cleanup")

## 1. Native Algorithm Introspection

In [ ]:
# Test: List native algorithms
native_algos = conn.algo.algorithms()

assert len(native_algos) > 0, "Should have native algorithms"
print(f"Algo 1.1 PASSED: {len(native_algos)} native Ryugraph algorithms available")
for algo in native_algos:
    print(f"  - {algo['name']}: {algo['description']}")

In [ ]:
# Test: Get algorithm info
pr_info = conn.algo.algorithm_info("pagerank")

assert pr_info is not None, "Should get algorithm info"
assert pr_info["name"] == "pagerank", f"Expected 'pagerank', got '{pr_info['name']}'"
assert "parameters" in pr_info, "Should have parameters"
assert "description" in pr_info, "Should have description"

print("Algo 1.2 PASSED: PageRank info:")
print(f"  Name: {pr_info['name']}")
print(f"  Category: {pr_info.get('category', 'N/A')}")
print(f"  Parameters: {[p['name'] for p in pr_info.get('parameters', [])]}")

## 2. Native Algorithm Execution

In [ ]:
# Test: Generic run() method for native algorithms
exec_result = conn.algo.run(
    algorithm="wcc",
    node_label="Person",
    property_name="algo_generic_wcc",
    edge_type="KNOWS"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
assert exec_result.nodes_updated == EXPECTED_NODE_COUNT

print(f"Algo 2.1 PASSED: Generic run() status={exec_result.status}, nodes={exec_result.nodes_updated}")

In [ ]:
# Test: PageRank execution
exec_result = conn.algo.pagerank(
    node_label="Person",
    property_name="algo_pr",
    edge_type="KNOWS"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
assert exec_result.nodes_updated == EXPECTED_NODE_COUNT, \
    f"Expected {EXPECTED_NODE_COUNT} nodes updated, got {exec_result.nodes_updated}"

print(f"Algo 2.2 PASSED: PageRank status={exec_result.status}, nodes={exec_result.nodes_updated}")

In [ ]:
# Test: PageRank results are queryable and between 0 and 1
result = conn.query(
    """
    MATCH (p:Person)
    WHERE p.algo_pr IS NOT NULL
    RETURN p.name, p.algo_pr
    ORDER BY p.algo_pr DESC
    """
)

assert result.row_count == EXPECTED_NODE_COUNT, f"Expected {EXPECTED_NODE_COUNT} rows"

for row in result.rows:
    score = row[1]
    assert 0 < score < 1, f"PageRank score {score} should be between 0 and 1"

print("Algo 2.3 PASSED: PageRank scores verified (all between 0 and 1)")

In [ ]:
# Test: Connected Components
exec_result = conn.algo.connected_components(
    node_label="Person",
    property_name="algo_wcc",
    edge_type="KNOWS"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
assert exec_result.nodes_updated == EXPECTED_NODE_COUNT

# Verify single component (connected graph)
component_count = conn.query_scalar("MATCH (p:Person) RETURN count(DISTINCT p.algo_wcc)")
assert component_count == 1, f"Expected 1 component, got {component_count}"

print(f"Algo 2.4 PASSED: Connected Components: {component_count} component(s)")

In [ ]:
# Test: Louvain community detection
exec_result = conn.algo.louvain(
    node_label="Person",
    property_name="algo_community",
    edge_type="KNOWS"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
assert exec_result.nodes_updated == EXPECTED_NODE_COUNT

# Verify community IDs are assigned
result = conn.query(
    "MATCH (p:Person) WHERE p.algo_community IS NOT NULL RETURN p.name, p.algo_community"
)
assert result.row_count == EXPECTED_NODE_COUNT

print(f"Algo 2.5 PASSED: Louvain status={exec_result.status}")

In [ ]:
# Test: Strongly Connected Components (SCC)
exec_result = conn.algo.scc(
    node_label="Person",
    property_name="algo_scc",
    edge_type="KNOWS"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
assert exec_result.nodes_updated == EXPECTED_NODE_COUNT

# Verify results are queryable
result = conn.query("MATCH (p:Person) WHERE p.algo_scc IS NOT NULL RETURN count(p)")
assert result.rows[0][0] == EXPECTED_NODE_COUNT

print(f"Algo 2.6 PASSED: SCC status={exec_result.status}, nodes={exec_result.nodes_updated}")

In [ ]:
# Test: SCC Kosaraju
exec_result = conn.algo.scc_kosaraju(
    node_label="Person",
    property_name="algo_scc_ko",
    edge_type="KNOWS"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
assert exec_result.nodes_updated == EXPECTED_NODE_COUNT

print(f"Algo 2.7 PASSED: SCC Kosaraju status={exec_result.status}, nodes={exec_result.nodes_updated}")

In [ ]:
# Test: K-Core decomposition
exec_result = conn.algo.kcore(
    node_label="Person",
    property_name="algo_kcore",
    edge_type="KNOWS"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
assert exec_result.nodes_updated == EXPECTED_NODE_COUNT

# Verify k-degree values are assigned (should be >= 0)
result = conn.query("MATCH (p:Person) RETURN p.name, p.algo_kcore ORDER BY p.algo_kcore DESC")
for row in result.rows:
    assert row[1] >= 0, f"K-core degree should be >= 0, got {row[1]}"

print(f"Algo 2.8 PASSED: K-Core status={exec_result.status}, nodes={exec_result.nodes_updated}")

In [ ]:
# Test: Label Propagation community detection
exec_result = conn.algo.label_propagation(
    node_label="Person",
    property_name="algo_label_prop",
    edge_type="KNOWS",
    max_iterations=100
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
assert exec_result.nodes_updated == EXPECTED_NODE_COUNT

# Verify label IDs are assigned
result = conn.query(
    "MATCH (p:Person) WHERE p.algo_label_prop IS NOT NULL RETURN p.name, p.algo_label_prop"
)
assert result.row_count == EXPECTED_NODE_COUNT

print(f"Algo 2.9 PASSED: Label Propagation status={exec_result.status}, nodes={exec_result.nodes_updated}")

In [ ]:
# Test: Triangle Count
exec_result = conn.algo.triangle_count(
    node_label="Person",
    property_name="algo_triangles",
    edge_type="KNOWS"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
assert exec_result.nodes_updated == EXPECTED_NODE_COUNT

# Verify triangle counts are >= 0
result = conn.query(
    "MATCH (p:Person) WHERE p.algo_triangles IS NOT NULL RETURN p.name, p.algo_triangles ORDER BY p.algo_triangles DESC"
)
assert result.row_count == EXPECTED_NODE_COUNT
for row in result.rows:
    assert row[1] >= 0, f"Triangle count should be >= 0, got {row[1]}"

print(f"Algo 2.10 PASSED: Triangle Count status={exec_result.status}, nodes={exec_result.nodes_updated}")

In [ ]:
# Test: Shortest Path between two nodes
# First, get two node IDs to find path between
nodes = conn.query("MATCH (p:Person) RETURN p.id, p.name LIMIT 2")
assert nodes.row_count >= 2, "Need at least 2 nodes for shortest path test"

source_id = nodes.rows[0][0]
target_id = nodes.rows[1][0]
source_name = nodes.rows[0][1]
target_name = nodes.rows[1][1]

print(f"Finding shortest path from {source_name} (id={source_id}) to {target_name} (id={target_id})")

exec_result = conn.algo.shortest_path(
    source_id=source_id,
    target_id=target_id,
    relationship_types=["KNOWS"],
    max_depth=5
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
# Result should contain path information
assert exec_result.result is not None, "Shortest path should return result"

print(f"Algo 2.11 PASSED: Shortest Path status={exec_result.status}")
print(f"  Path result: {exec_result.result}")

## 3. NetworkX Algorithm Tests

In [ ]:
# Test: Filter algorithms by category
centrality_algos = conn.networkx.algorithms(category="centrality")

assert len(centrality_algos) > 0, "Should have centrality algorithms"
for algo in centrality_algos:
    assert algo["category"] == "centrality", f"Expected 'centrality', got '{algo['category']}'"

print(f"Algo 3.2 PASSED: Centrality algorithms: {len(centrality_algos)}")
print(f"  Examples: {[a['name'] for a in centrality_algos[:5]]}")

In [ ]:
# Test: NetworkX algorithm_info() for algorithm details
bc_info = conn.networkx.algorithm_info("betweenness_centrality")

assert bc_info is not None, "Should get algorithm info"
assert bc_info["name"] == "betweenness_centrality", f"Expected 'betweenness_centrality', got '{bc_info['name']}'"
assert "description" in bc_info, "Should have description"

print("Algo 3.2b PASSED: NetworkX algorithm_info():")
print(f"  Name: {bc_info['name']}")
print(f"  Category: {bc_info.get('category', 'N/A')}")
print(f"  Description: {bc_info.get('description', 'N/A')[:80]}...")

In [ ]:
# Test: Betweenness Centrality
exec_result = conn.networkx.betweenness_centrality(
    node_label="Person",
    property_name="algo_bc"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
assert exec_result.nodes_updated == EXPECTED_NODE_COUNT

# Verify results are >= 0
result = conn.query(
    "MATCH (p:Person) WHERE p.algo_bc IS NOT NULL RETURN p.name, p.algo_bc ORDER BY p.algo_bc DESC"
)
for row in result.rows:
    assert row[1] >= 0, f"Betweenness should be >= 0, got {row[1]}"

print(f"Algo 3.3 PASSED: Betweenness Centrality status={exec_result.status}")

In [ ]:
# Test: Degree Centrality
exec_result = conn.networkx.degree_centrality(
    node_label="Person",
    property_name="algo_dc"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"

# Verify scores are between 0 and 1 (normalized)
result = conn.query("MATCH (p:Person) RETURN p.name, p.algo_dc ORDER BY p.algo_dc DESC")
for row in result.rows:
    score = row[1]
    assert 0 <= score <= 1, f"Degree centrality should be 0-1, got {score}"

print(f"Algo 3.4 PASSED: Degree Centrality status={exec_result.status}")

In [ ]:
# Test: Closeness Centrality
exec_result = conn.networkx.closeness_centrality(
    node_label="Person",
    property_name="algo_cc"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
print(f"Algo 3.5 PASSED: Closeness Centrality status={exec_result.status}")

In [ ]:
# Test: Eigenvector Centrality
exec_result = conn.networkx.eigenvector_centrality(
    node_label="Person",
    property_name="algo_ev",
    max_iter=100
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"

# Verify results are >= 0
result = conn.query(
    "MATCH (p:Person) WHERE p.algo_ev IS NOT NULL RETURN p.name, p.algo_ev ORDER BY p.algo_ev DESC"
)
assert result.row_count == EXPECTED_NODE_COUNT
for row in result.rows:
    assert row[1] >= 0, f"Eigenvector centrality should be >= 0, got {row[1]}"

print(f"Algo 3.5b PASSED: Eigenvector Centrality status={exec_result.status}")

In [ ]:
# Test: Clustering Coefficient
exec_result = conn.networkx.clustering_coefficient(
    node_label="Person",
    property_name="algo_clustering"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"

# Verify results are between 0 and 1
result = conn.query(
    "MATCH (p:Person) WHERE p.algo_clustering IS NOT NULL RETURN p.name, p.algo_clustering ORDER BY p.algo_clustering DESC"
)
assert result.row_count == EXPECTED_NODE_COUNT
for row in result.rows:
    score = row[1]
    assert 0 <= score <= 1, f"Clustering coefficient should be 0-1, got {score}"

print(f"Algo 3.5c PASSED: Clustering Coefficient status={exec_result.status}")

In [ ]:
# Test: Generic run() method for NetworkX
exec_result = conn.networkx.run(
    algorithm="degree_centrality",
    node_label="Person",
    property_name="algo_nx_generic_dc"
)

assert exec_result.status == "completed", f"Expected 'completed', got '{exec_result.status}'"
print(f"Algo 3.6 PASSED: Generic NetworkX run() status={exec_result.status}")

In [ ]:
# Test: Multiple algorithm results coexist
count = conn.query_scalar(
    """
    MATCH (p:Person)
    WHERE p.algo_pr IS NOT NULL AND p.algo_bc IS NOT NULL AND p.algo_dc IS NOT NULL
    RETURN count(p)
    """
)

assert count == EXPECTED_NODE_COUNT, f"Expected all {EXPECTED_NODE_COUNT} nodes to have all properties"
print(f"Algo 3.7 PASSED: All {count} nodes have multiple algorithm results")

In [ ]:
# Test: Verify all centrality measures together
df = conn.query_df("""
    MATCH (p:Person)
    RETURN 
        p.name AS name,
        p.algo_pr AS pagerank,
        p.algo_bc AS betweenness,
        p.algo_dc AS degree,
        p.algo_cc AS closeness
    ORDER BY p.algo_pr DESC
""")

assert len(df) == EXPECTED_NODE_COUNT

# Verify no nulls
for col in ['pagerank', 'betweenness', 'degree', 'closeness']:
    nulls = df[col].null_count()
    assert nulls == 0, f"Column '{col}' has {nulls} null values"

print("Algo 3.8 PASSED: All centrality measures computed:")
print(df)

## 4. Lock Status Tests

In [ ]:
# Test: Get lock status when unlocked
lock = conn.get_lock()

assert lock is not None, "Lock status should not be None"
assert hasattr(lock, 'locked'), "Lock should have 'locked' attribute"
assert not lock.locked, f"Expected unlocked, got locked={lock.locked}"

print(f"Algo 4.1 PASSED: Lock status: locked={lock.locked}")

In [ ]:
# Test: Lock released after algorithm completion
execution = conn.algo.pagerank(
    node_label="Person",
    property_name="algo_lock_test_pr",
    edge_type="KNOWS"
)
assert execution.status == "completed"

# Check lock is released
lock = conn.get_lock()
assert not lock.locked, "Lock should be released after algorithm completes"

print("Algo 4.2 PASSED: Lock correctly released after algorithm")

In [ ]:
# Test: Lock status fields
lock = conn.get_lock()

# When unlocked, holder info should be None/empty
if hasattr(lock, "holder_name"):
    assert lock.holder_name is None or lock.holder_name == "", \
        f"Expected empty holder_name when unlocked, got '{lock.holder_name}'"
if hasattr(lock, "algorithm"):
    assert lock.algorithm is None or lock.algorithm == "", \
        f"Expected empty algorithm when unlocked, got '{lock.algorithm}'"

print("Algo 4.3 PASSED: Lock status fields verified")

## Summary

In [ ]:
# Cleanup is handled automatically by ctx via atexit
# For interactive use, you can call ctx.cleanup() manually

print("Algorithm tests completed - cleanup will happen automatically")

In [ ]:
# Algorithm tests completed
print("Algorithm tests completed")

In [ ]:
print("\n" + "="*60)
print("ALGORITHM E2E TESTS COMPLETED!")
print("="*60)
print("\nValidated:")
print("  1. Native Algorithm Introspection:")
print("    - algorithms() listing")
print("    - algorithm_info() details")
print("  2. Native Algorithm Execution:")
print("    - Generic run() method")
print("    - PageRank")
print("    - Connected Components (WCC)")
print("    - Louvain community detection")
print("    - Strongly Connected Components (SCC)")
print("    - SCC Kosaraju")
print("    - K-Core decomposition")
print("  3. NetworkX Algorithms:")
print("    - Algorithm listing and filtering")
print("    - Betweenness Centrality")
print("    - Degree Centrality")
print("    - Closeness Centrality")
print("    - Generic run() method")
print("    - Multiple results coexistence")
print("  4. Lock Status:")
print("    - get_lock() when unlocked")
print("    - Lock release after completion")
print("    - Lock status fields")
print("\nAll resources will be cleaned up automatically via atexit")